# Ejercicio 5: Espacio Vectorial

## Objetivo de la práctica
- Implementar un Sistema de Recuperación de Información completo, desde la lectura del corpus hasta la recuperación de resultados.

### Autor: Pérez Pineda Andrés Alejandro

In [20]:
import math
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

stop_words = set(ENGLISH_STOP_WORDS)

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

def limpiar_texto(texto):
    if not isinstance(texto, str):
        return ""
    # Convierte a minúsculas
    texto = texto.lower()
    # Eliminar puntuación y caracteres especiales
    texto = re.sub(r'[^a-z0-9\s]', '', texto)
    # Tokenizar dividiendo por espacios
    tokens = texto.split()
    # Filtrar stopwords
    tokens_limpios = [word for word in tokens if word not in stop_words]
    return " ".join(tokens_limpios)

print("Importaciones y configuración completadas")

/kaggle/input/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects/wikipedia_text_corpus.csv
Importaciones y configuración completadas


## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus
2. Realiza las etapas de preprocesamiento sobre el corpus


In [21]:
ruta_archivo = '/kaggle/input/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects/wikipedia_text_corpus.csv' 
df = pd.read_csv(ruta_archivo)

# df = df.sample(5000, random_state=42)

print(df.head())

# Preprocesamiento:
def limpiar_texto(texto):
    if not isinstance(texto, str):
        return ""
    # Convierte a minúsculas
    texto = texto.lower()
    # Elimina las puntuaciones y caracteres especiales
    texto = re.sub(r'[^a-z0-9\s]', '', texto)
    # Tokeniza el documento dividiendo por espacios
    tokens = texto.split()
    # Filtra stopwords
    tokens_limpios = [word for word in tokens if word not in stop_words]
    return " ".join(tokens_limpios)

df['texto_limpio'] = df['text'].apply(limpiar_texto)

print("\nCorpus cargado y preprocesado")

   Unnamed: 0                                               text
0           1  Anovo\n\nAnovo (formerly A Novo) is a computer...
1           2  Battery indicator\n\nA battery indicator (also...
2           3  Bob Pease\n\nRobert Allen Pease (August 22, 19...
3           4  CAVNET\n\nCAVNET was a secure military forum w...
4           5  CLidar\n\nThe CLidar is a scientific instrumen...

Corpus cargado y preprocesado


## Parte 1: Recuperación con TF-IDF

### Actividad:
3. Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF
4. A partir de un conjunto de 10 queries, verifica la recuperación del sistema

In [22]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df['texto_limpio'])

def buscar_tfidf(query, top_k=3):
    # Preprocesa la query de la misma forma que el corpus
    query_limpia = limpiar_texto(query)
    query_vec = tfidf_vectorizer.transform([query_limpia])
    
    # Calcula la similitud coseno entre la query y todos los documentos
    similitudes = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # Obtiene los índices de los documentos con mayor similitud
    indices_top = similitudes.argsort()[::-1][:top_k]
    
    resultados = []
    for idx in indices_top:
        resultados.append({
            'index': idx,
            'title': df['title'].iloc[idx] if 'title' in df.columns else f"Doc {idx}",
            'score': similitudes[idx]
        })
    return resultados

# Consultas:
queries = [
    "artificial intelligence and machine learning",
    "history of ancient rome and emperors",
    "quantum physics and entanglement",
    "global warming climate change effects",
    "space exploration and mars missions",
    "world war two major battles",
    "renewable energy solar panels",
    "human genome project genetics",
    "financial markets and cryptocurrency",
    "famous Renaissance painters and art"
]

print("Recuperación TF-IDF")
for q in queries[:10]:
    print(f"\nQuery: '{q}'")
    res = buscar_tfidf(q, top_k=5)
    for r in res:
        print(f"  -> Título: {r['title']} (Score: {r['score']:.4f})")

Recuperación TF-IDF

Query: 'artificial intelligence and machine learning'
  -> Título: Doc 10547 (Score: 0.6562)
  -> Título: Doc 5229 (Score: 0.4363)
  -> Título: Doc 7900 (Score: 0.4036)
  -> Título: Doc 3905 (Score: 0.3420)
  -> Título: Doc 6853 (Score: 0.3336)

Query: 'history of ancient rome and emperors'
  -> Título: Doc 2536 (Score: 0.1804)
  -> Título: Doc 6079 (Score: 0.1642)
  -> Título: Doc 3359 (Score: 0.1548)
  -> Título: Doc 2854 (Score: 0.1536)
  -> Título: Doc 8776 (Score: 0.1369)

Query: 'quantum physics and entanglement'
  -> Título: Doc 4442 (Score: 0.2706)
  -> Título: Doc 2653 (Score: 0.2586)
  -> Título: Doc 9412 (Score: 0.2308)
  -> Título: Doc 5956 (Score: 0.2300)
  -> Título: Doc 9854 (Score: 0.2281)

Query: 'global warming climate change effects'
  -> Título: Doc 941 (Score: 0.3038)
  -> Título: Doc 10805 (Score: 0.2387)
  -> Título: Doc 2547 (Score: 0.2141)
  -> Título: Doc 311 (Score: 0.2026)
  -> Título: Doc 7289 (Score: 0.2022)

Query: 'space exploration 

## Parte 2: Recuperación con BM25

### Actividad:
5. Implementa un sistema de recuperación usando el modelo BM25.
6. Para el mismo conjunto de 10 queries, verifica la recuperación del sistema

In [23]:
class BM25:
    def __init__(self, corpus_tokenizado, k1=1.5, b=0.75):
        
        self.k1 = k1
        self.b = b
        self.N = len(corpus_tokenizado)
        self.avgdl = sum(len(doc) for doc in corpus_tokenizado) / self.N
        
        self.doc_lengths = [len(doc) for doc in corpus_tokenizado]
        self.doc_tf = []
        self.df = {}
        self.idf = {}
        
        # Precalculamiento de frecuencias (TF) y frecuencias de documento (DF)
        for doc in corpus_tokenizado:
            tf_dict = {}
            for term in doc:
                tf_dict[term] = tf_dict.get(term, 0) + 1
            self.doc_tf.append(tf_dict)
            
            for term in set(doc):
                self.df[term] = self.df.get(term, 0) + 1
                
        # Precalculamiento de IDF con suavizado RSJ (corrección de 0.5)
        for term, df_t in self.df.items():
            idf_val = math.log((self.N - df_t + 0.5) / (df_t + 0.5))
            self.idf[term] = max(0.0, idf_val)

    def get_scores(self, query_tokenizada):
        # Calcula los scores de los documentos
        scores = np.zeros(self.N)
        for term in query_tokenizada:
            if term not in self.idf:
                continue
            
            term_idf = self.idf[term]
            for doc_idx in range(self.N):
                tf = self.doc_tf[doc_idx].get(term, 0)
                if tf == 0:
                    continue
                
                doc_len = self.doc_lengths[doc_idx]
                
                norm_len = (1.0 - self.b) + self.b * (doc_len / self.avgdl)
                
                numerador = term_idf * tf * (self.k1 + 1.0)
                denominador = tf + self.k1 * norm_len
                
                scores[doc_idx] += numerador / denominador
                
        return scores

corpus_para_bm25 = [texto.split() for texto in df['texto_limpio']]

bm25_modelo = BM25(corpus_para_bm25, k1=1.5, b=0.75)

def buscar_bm25(query, top_k=3):
    query_tokens = limpiar_texto(query).split()
    
    scores = bm25_modelo.get_scores(query_tokens)
    indices_top = scores.argsort()[::-1][:top_k]
    
    resultados = []
    for idx in indices_top:
        resultados.append({
            'index': idx,
            'title': df['title'].iloc[idx] if 'title' in df.columns else f"Doc {idx}",
            'score': scores[idx]
        })
    return resultados

print("Recuperación BM25")
for q in queries[:10]:
    print(f"\nQuery: '{q}'")
    res = buscar_bm25(q, top_k=5)
    for r in res:
        print(f"  -> Título: {r['title']} (Score: {r['score']:.4f})")

Recuperación BM25

Query: 'artificial intelligence and machine learning'
  -> Título: Doc 10547 (Score: 23.9319)
  -> Título: Doc 8528 (Score: 21.2229)
  -> Título: Doc 5991 (Score: 20.9625)
  -> Título: Doc 9538 (Score: 20.8776)
  -> Título: Doc 6569 (Score: 20.7331)

Query: 'history of ancient rome and emperors'
  -> Título: Doc 2536 (Score: 18.6567)
  -> Título: Doc 8776 (Score: 17.4407)
  -> Título: Doc 2854 (Score: 14.8995)
  -> Título: Doc 1932 (Score: 14.8195)
  -> Título: Doc 1400 (Score: 14.5194)

Query: 'quantum physics and entanglement'
  -> Título: Doc 1741 (Score: 19.3093)
  -> Título: Doc 9447 (Score: 15.8729)
  -> Título: Doc 4442 (Score: 15.8526)
  -> Título: Doc 3099 (Score: 14.7918)
  -> Título: Doc 6307 (Score: 14.3522)

Query: 'global warming climate change effects'
  -> Título: Doc 941 (Score: 22.7829)
  -> Título: Doc 4139 (Score: 22.4578)
  -> Título: Doc 954 (Score: 19.7340)
  -> Título: Doc 3781 (Score: 18.6887)
  -> Título: Doc 4978 (Score: 18.4362)

Query: 's

## Parte 3: Comparación de resultados

### Actividad:
7. Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación 

In [24]:
print(" TF-IDF vs. BM25 ")

k_docs = 10

for i, q in enumerate(queries, 1):
    print(f"\n[{i}] Query: '{q}'")
    
    # Obtención de resultados en bruto
    res_tfidf = buscar_tfidf(q, top_k=k_docs)
    res_bm25 = buscar_bm25(q, top_k=k_docs)
    
    if len(res_bm25) > 0 and res_bm25[0]['score'] > 0:
        max_score_bm25 = res_bm25[0]['score']
    else:
        max_score_bm25 = 1.0
        
    titulos_tfidf = []
    porcentajes_tfidf = []
    titulos_bm25 = []
    porcentajes_bm25 = []
    
    for r_tf, r_bm in zip(res_tfidf, res_bm25):
        # TF-IDF: Similitud Coseno directa a porcentaje
        sim_tfidf_pct = r_tf['score'] * 100.0
        titulos_tfidf.append(r_tf['title'])
        porcentajes_tfidf.append(f"{sim_tfidf_pct:.2f}%")
        
        # BM25: Relevancia relativa al Top 1, ya que este tendrá el 100% de simlitud
        sim_bm25_pct = (r_bm['score'] / max_score_bm25) * 100.0
        titulos_bm25.append(r_bm['title'])
        porcentajes_bm25.append(f"{sim_bm25_pct:.2f}%")
        
    etiquetas_ranking = [f"Top {j}" for j in range(1, len(titulos_tfidf) + 1)]
    
    df_porcentajes = pd.DataFrame({
        'Ranking': etiquetas_ranking,
        'TF-IDF Documento': titulos_tfidf,
        'Similitud TF-IDF': porcentajes_tfidf,
        'BM25 Documento': titulos_bm25,
        'Relevancia BM25': porcentajes_bm25
    })
    
    print(df_porcentajes.to_string(index=False))
    print("-" * 85)

 TF-IDF vs. BM25 

[1] Query: 'artificial intelligence and machine learning'
Ranking TF-IDF Documento Similitud TF-IDF BM25 Documento Relevancia BM25
  Top 1        Doc 10547           65.62%      Doc 10547         100.00%
  Top 2         Doc 5229           43.63%       Doc 8528          88.68%
  Top 3         Doc 7900           40.36%       Doc 5991          87.59%
  Top 4         Doc 3905           34.20%       Doc 9538          87.24%
  Top 5         Doc 6853           33.36%       Doc 6569          86.63%
  Top 6         Doc 4733           33.20%      Doc 10699          86.39%
  Top 7         Doc 4563           32.79%        Doc 466          79.13%
  Top 8         Doc 8060           31.58%       Doc 1075          78.43%
  Top 9         Doc 9239           30.88%       Doc 9105          78.16%
 Top 10          Doc 485           29.77%        Doc 485          76.54%
-------------------------------------------------------------------------------------

[2] Query: 'history of ancient ro